In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import random
from scipy import stats
from scipy.stats import fisher_exact
from scipy.stats.contingency import expected_freq

from localcider.sequenceParameters import SequenceParameters
import umap
MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}#from AAindex MUNV940103
hTMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')
mdSAMs = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
mdSAMs = mdSAMs.drop_duplicates('AAseq')

In [ ]:
df_PCA = pd.read_csv('GPCR_ThT/RNAseq/ywang34_data/PCA_data.tsv',delimiter='\t')

In [ ]:
df_PCA['label'] = ''
df_PCA.iloc[0:3,-1]= '$WT$'
df_PCA.iloc[3:6,-1]= '$V451P$'
df_PCA.iloc[6:,-1]= '$pCDNA$'

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5)) 
sns.scatterplot(x='PC1',y='PC2',data=df_PCA,hue='label',s=100,palette=['#C03830','#B4B5B4','#000080'])
plt.xlabel('$PC1\,(60.7$%$)$',fontsize=10)
plt.ylabel('$PC2\,(10.5$%$)$',fontsize=10)
ax.grid()
plt.legend(fontsize=8)
plt.title('$RNA‑seq\;PCA$')
plt.savefig('PCA_ywang34.svg',bbox_inches='tight',transparent=True)

In [ ]:
df = pd.read_csv('GPCR_ThT/RNAseq/ywang34_data/deseq2_HTR7_MUT_vs_HTR7_WT.tsv',delimiter='\t')

In [ ]:
from adjustText import adjust_text

In [ ]:
df['padj_log'] = -1*np.log10(df['padj'])

In [ ]:
df[(df.padj_log>1.3)&(df.log2FoldChange>.4)]

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
ax.grid()
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df,edgecolor=None,color='#B4B5B4')
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df[(df.padj_log>2)&(df.log2FoldChange>.4)],edgecolor=None,color='#18827E')
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df[(df.padj_log>2)&(df.log2FoldChange<-.4)],edgecolor=None,color='#B34F91')
ax.set_xlim([-1,1])
ax.set_ylim([-0.4,5.5])
texts = []
for i in df[(df.padj_log>2)&(df.log2FoldChange>.4)].index:
    texts.append(plt.text(df.loc[i,'log2FoldChange'],df.loc[i,'padj_log'],
            '$%s$'%df.loc[i,'geneSymbol'],
                          fontsize=6))
adjust_text(texts, autoalign='xy',force_text=(-.2, .1),force_points=(-.1, 0.1),
            only_move={'points':'xy', 'text':'xy'},
            arrowprops=dict(arrowstyle="-", color='k', lw=0.5),zorder=0)
texts = []
for i in df[(df.padj_log>2)&(df.log2FoldChange<-.4)].index:
    texts.append(plt.text(df.loc[i,'log2FoldChange'],df.loc[i,'padj_log'],
            '$%s$'%df.loc[i,'geneSymbol'],
                          fontsize=6))
adjust_text(texts, autoalign='xy',force_text=(0.0),force_points=(-0, 0.0),
            only_move={'points':'xy', 'text':'xy'},
            arrowprops=dict(arrowstyle="-", color='k', lw=0.5),zorder=0)
plt.title('$V451P\;vs\;WT$',fontsize=10)
plt.xlabel('$FC(log_{2})$',fontsize=10)
plt.ylabel('$FDR(‑log_{10})$',fontsize=10)
plt.axvline(.4,color='k',ls='--',lw=.5)
plt.axvline(-.4,color='k',ls='--',lw=.5)
plt.axhline(2,color='k',ls='--',lw=.5)
plt.savefig('volcano_ywang34.svg',bbox_inches='tight',transparent=True)

In [ ]:
df[(df.padj_log>2)&(df.log2FoldChange>.4)]

In [ ]:
df_WT_control = pd.read_csv('GPCR_ThT/RNAseq/ywang34_data/deseq2_HTR7_WT_vs_HEK_pCDNA.tsv',delimiter='\t')
df_WT_control['padj_log'] = -1*np.log10(df_WT_control['padj'])
df_MUT_control = pd.read_csv('GPCR_ThT/RNAseq/ywang34_data/deseq2_HTR7_MUT_vs_HEK_pCDNA.tsv',delimiter='\t')
df_MUT_control['padj_log'] = -1*np.log10(df_MUT_control['padj'])

In [ ]:
df_WT_control[df_WT_control.geneSymbol=='JUNB']
df_MUT_control[df_MUT_control.geneSymbol=='JUNB']

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
ax.grid()
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df_WT_control,edgecolor=None,color='#B4B5B4')
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df_WT_control[(df_WT_control.padj_log>2)&(df_WT_control.log2FoldChange>1)],edgecolor=None,color='#18827E')
plt.title('$WT\;vs\;pCDNA$',fontsize=10)
plt.xlabel('$FC(log_{2})$',fontsize=10)
plt.ylabel('$FDR(‑log_{10})$',fontsize=10)
plt.axvline(1,color='k',ls='--',lw=.5)
plt.axhline(2,color='k',ls='--',lw=.5)
plt.savefig('volcano_ywang34_WT_vs_pCDNA.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
ax.grid()
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df_MUT_control,edgecolor=None,color='#B4B5B4')
sns.scatterplot(x='log2FoldChange',y='padj_log',data=df_MUT_control[(df_MUT_control.padj_log>2)&(df_MUT_control.log2FoldChange>1)],edgecolor=None,color='#18827E')
plt.title('$V451P\;vs\;pCDNA$',fontsize=10)
plt.xlabel('$FC(log_{2})$',fontsize=10)
plt.ylabel('$FDR(‑log_{10})$',fontsize=10)
plt.axvline(1,color='k',ls='--',lw=.5)
plt.axhline(2,color='k',ls='--',lw=.5)
plt.savefig('volcano_ywang34_MUT_vs_pCDNA.svg',bbox_inches='tight',transparent=True)

In [ ]:
from matplotlib_venn import venn2

import matplotlib.pyplot as plt


# Example lists
list1 = list(set(df_MUT_control[(df_MUT_control.padj_log>2)&(df_MUT_control.log2FoldChange>1)].geneSymbol))
list2 =list(set(df_WT_control[(df_WT_control.padj_log>2)&(df_WT_control.log2FoldChange>1)].geneSymbol))
# Create a Venn diagram
set1 = set(list1)
set2 = set(list2)

# Create a Venn diagram
venn2([set1, set2], set_labels=("V451P", "$WT$"),set_colors=('#B4B5B4','#C03830'),alpha=1,)

# Display the plot
plt.title("Venn Diagram of Two Lists")
plt.savefig('volcano_ywang34_MUT_vs_pCDNA_venn.svg',bbox_inches='tight',transparent=True)

In [ ]:
set.intersection(set1,set2)

In [ ]:
SSTM_Cytoplasmic = pd.read_pickle('SSTM_Cytoplasmic.pkl')
SSTM_membrane_boundery = pd.read_pickle('SSTM_membrane_boundery.pkl')
sstm_acc  = AF2[AF2.ENSG.isin(SSTM_Cytoplasmic.ENSG)].acc
iPTM = pd.read_pickle('iPTM_SSTMs_v2.pkl')
iPTM = iPTM.drop(index = iPTM[pd.isna(iPTM.site)].index)
for i in iPTM.index:
    iPTM.loc[i,'site_int'] = int(iPTM.loc[i,'site'][1:])

In [ ]:
sstm_acc

In [ ]:
TH = 3
acc_with_iptms = set(iPTM[(iPTM.acc.isin(sstm_acc))&(iPTM.ptm_type=='Phosphorylation')&(iPTM.score>=TH)].acc)

In [ ]:
len(set.intersection(set(acc_with_iptms),set(mdSAMs.acc)))

In [ ]:
acc_with_iptms